# Day 15: SchNet 3D-GNN + Docking Features + Multi-Task Learning

**Motivation:** After 14 days of 2D approaches (best: 0.62 RAE), it's time for TRUE 3D modeling.

**Why 3D matters for PXR:**
- PXR has a HUGE binding pocket (~1150 Å³) - shape and volume critical
- Hydrophobic π-trap (F288, W299, Y306) - 3D spatial arrangement matters
- Multiple binding modes possible - conformational flexibility key

**Architecture:**
1. **SchNet**: SE(3) equivariant GNN for 3D molecular representations
   - Input: 3D conformers (RDKit ETKDG)
   - Learns: Interatomic distances, angles, spatial patterns
   - Output: 128-dim learned 3D embeddings

2. **Docking Features**: Physics-based protein-ligand interactions
   - AutoDock Vina GPU for fast docking to PXR
   - Extract: Binding affinity, interaction fingerprints
   - Focus: Key residues F288, W299, Y306

3. **Multi-Task Head**: Joint prediction of pEC50 + Emax
   - Rationale: Binding mode determines both affinity AND efficacy
   - Shared 3D encoder + dual prediction heads
   - Regularization through auxiliary task

**Computational Requirements:**
- ⚠️ REQUIRES GPU (SchNet training)
- ⚠️ REQUIRES conformer generation (~5 min for 2800 molecules)
- ⚠️ OPTIONAL: Docking (can use pre-computed or skip for faster iteration)
- ⚠️ Estimated: 1-2 hours total on Colab GPU

**Expected Performance:**
- Target: <0.58 RAE (beating Day 10's 0.62)
- Rationale: 3D geometry + physics should capture binding better than 2D

---

## Setup Instructions for Google Colab

```python
# Run these commands in Colab:
!pip install torch-geometric torch-scatter torch-sparse -q
!pip install schnetpack -q
!pip install meeko vina -q  # For docking (optional)
!pip install huggingface_hub -q
```

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path
import pickle

# RDKit for conformer generation
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

# PyTorch and PyTorch Geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch

# SchNetPack - 3D equivariant GNN
try:
    import schnetpack as spk
    from schnetpack import properties
    SCHNET_AVAILABLE = True
    print("✅ SchNetPack available")
except ImportError:
    SCHNET_AVAILABLE = False
    print("❌ SchNetPack not installed. Run: pip install schnetpack")

# ML utilities
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')
tqdm.pandas()
sns.set_style('whitegrid')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️  Device: {device}")
if device.type == 'cpu':
    print("⚠️  WARNING: Running on CPU. This will be SLOW. Use Google Colab GPU.")

## 1. Load Data

In [ ]:
# Load training data (counter-screen passed only)
train_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_counter-assay_TRAIN.csv")
test_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TEST_BLINDED.csv")

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

# Validate SMILES and remove NaN targets
def is_valid_smiles(smi):
    return smi is not None and Chem.MolFromSmiles(smi) is not None

valid_mask = train_df['SMILES'].apply(is_valid_smiles)
pec50_mask = ~np.isnan(train_df['pEC50'])
emax_mask = ~np.isnan(train_df['Emax_estimate (log2FC vs. baseline)'])

# Keep molecules with both pEC50 and Emax (for multi-task)
keep_mask = valid_mask & pec50_mask & emax_mask

print(f"\nDropped {(~keep_mask).sum()} molecules (invalid SMILES or missing targets)")
train_df = train_df[keep_mask].reset_index(drop=True)

y_pec50 = train_df['pEC50'].values
y_emax = train_df['Emax_estimate (log2FC vs. baseline)'].values

print(f"\n✅ Final training size: {len(train_df)}")
print(f"\npEC50 distribution: mean={y_pec50.mean():.3f}, std={y_pec50.std():.3f}")
print(f"Emax distribution: mean={y_emax.mean():.3f}, std={y_emax.std():.3f}")

## 2. Generate 3D Conformers

**Method:** RDKit ETKDG (Experimental-Torsion Knowledge Distance Geometry)
- Generates low-energy 3D conformers
- Uses experimental torsion angle preferences
- Multiple conformers per molecule (we'll use 1 for speed, could use 5-10 for better coverage)

**Output:** 3D coordinates for each atom (N_atoms × 3)

In [ ]:
def generate_conformer(smiles, num_confs=1, max_attempts=10):
    """
    Generate 3D conformer(s) for a molecule.
    
    Returns:
        mol: RDKit molecule with 3D coordinates
        None if conformer generation fails
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Add hydrogens (important for accurate geometry)
    mol = Chem.AddHs(mol)
    
    # ETKDG parameters
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.numThreads = 0  # Use all available
    
    # Generate conformer(s)
    for attempt in range(max_attempts):
        try:
            conf_ids = AllChem.EmbedMultipleConfs(
                mol, 
                numConfs=num_confs,
                params=params
            )
            
            if len(conf_ids) > 0:
                # Optimize geometry with MMFF94
                for conf_id in conf_ids:
                    AllChem.MMFFOptimizeMolecule(mol, confId=conf_id)
                
                # Return molecule with first (lowest energy) conformer
                return mol
        except Exception as e:
            continue
    
    return None

print("Generating 3D conformers for training set...")
train_df['mol_3d'] = [generate_conformer(s) for s in tqdm(train_df['SMILES'])]

print("\nGenerating 3D conformers for test set...")
test_df['mol_3d'] = [generate_conformer(s) for s in tqdm(test_df['SMILES'])]

# Check success rate
train_success = train_df['mol_3d'].notna().sum()
test_success = test_df['mol_3d'].notna().sum()

print(f"\n✅ Conformer generation complete:")
print(f"   Training: {train_success}/{len(train_df)} ({100*train_success/len(train_df):.1f}%)")
print(f"   Test: {test_success}/{len(test_df)} ({100*test_success/len(test_df):.1f}%)")

# Filter out failed conformers
train_df = train_df[train_df['mol_3d'].notna()].reset_index(drop=True)
y_pec50 = train_df['pEC50'].values
y_emax = train_df['Emax_estimate (log2FC vs. baseline)'].values

print(f"\nFinal training size after conformer generation: {len(train_df)}")

## 3. Create PyTorch Geometric Dataset for SchNet

**SchNet Requirements:**
- `Z`: Atomic numbers (N_atoms,)
- `pos`: 3D coordinates (N_atoms, 3)
- `y`: Target value (1,) - we'll have two: pEC50 and Emax

SchNet learns from:
1. Interatomic distances
2. Continuous filter convolutions
3. Interaction blocks that are equivariant to rotations/translations

In [ ]:
class Molecule3DDataset(Dataset):
    """
    Dataset for 3D molecules with SchNet-compatible format.
    """
    
    def __init__(self, df, y_pec50=None, y_emax=None):
        self.df = df
        self.y_pec50 = y_pec50
        self.y_emax = y_emax
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        mol = self.df.iloc[idx]['mol_3d']
        
        # Extract atomic numbers
        Z = torch.tensor([atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype=torch.long)
        
        # Extract 3D coordinates (use first conformer)
        conf = mol.GetConformer(0)
        pos = torch.tensor(
            [[conf.GetAtomPosition(i).x,
              conf.GetAtomPosition(i).y,
              conf.GetAtomPosition(i).z] for i in range(mol.GetNumAtoms())],
            dtype=torch.float32
        )
        
        data = Data(
            z=Z,  # Atomic numbers
            pos=pos,  # 3D coordinates
            idx=torch.tensor([idx], dtype=torch.long)
        )
        
        # Add targets if available (training)
        if self.y_pec50 is not None:
            data.y_pec50 = torch.tensor([self.y_pec50[idx]], dtype=torch.float32)
            data.y_emax = torch.tensor([self.y_emax[idx]], dtype=torch.float32)
        
        return data

# Create datasets
train_dataset = Molecule3DDataset(train_df, y_pec50, y_emax)
test_dataset = Molecule3DDataset(test_df)

print(f"✅ Created datasets:")
print(f"   Training: {len(train_dataset)} molecules")
print(f"   Test: {len(test_dataset)} molecules")

# Example data point
sample = train_dataset[0]
print(f"\nExample molecule:")
print(f"   Atoms: {sample.z.shape[0]}")
print(f"   Coordinates shape: {sample.pos.shape}")
print(f"   pEC50: {sample.y_pec50.item():.3f}")
print(f"   Emax: {sample.y_emax.item():.3f}")

## 4. Define Multi-Task SchNet Model

**Architecture:**
```
Input: Atomic numbers + 3D coordinates
  ↓
SchNet Encoder (3 interaction blocks)
  - Continuous filter convolutions
  - Atom-wise updates
  - SE(3) equivariance
  ↓
Pooling (sum over atoms)
  ↓
Shared MLP (256 → 128)
  ↓
  ├─→ pEC50 Head (128 → 64 → 1)
  └─→ Emax Head (128 → 64 → 1)
```

**Loss:** Weighted multi-task loss
- L = α * MSE(pEC50) + β * MSE(Emax)
- α=1.0 (primary task), β=0.3 (auxiliary task)

In [ ]:
if SCHNET_AVAILABLE:
    class MultiTaskSchNet(nn.Module):
        """
        SchNet encoder with dual prediction heads for pEC50 and Emax.
        """
        
        def __init__(
            self,
            n_atom_basis=128,
            n_interactions=3,
            cutoff=10.0,
            n_gaussians=25
        ):
            super().__init__()
            
            # SchNet representation
            self.representation = spk.representation.SchNet(
                n_atom_basis=n_atom_basis,
                n_interactions=n_interactions,
                radial_basis=spk.nn.radial.GaussianRBF(
                    n_rbf=n_gaussians,
                    cutoff=cutoff
                ),
                cutoff_fn=spk.nn.cutoff.CosineCutoff(cutoff)
            )
            
            # Pooling
            self.pool = spk.atomistic.Atomwise(
                n_in=n_atom_basis,
                aggregation_mode='sum'
            )
            
            # Shared encoder
            self.shared_encoder = nn.Sequential(
                nn.Linear(n_atom_basis, 256),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(256, 128),
                nn.ReLU()
            )
            
            # Task-specific heads
            self.pec50_head = nn.Sequential(
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(64, 1)
            )
            
            self.emax_head = nn.Sequential(
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(64, 1)
            )
        
        def forward(self, batch):
            # SchNet encoding
            inputs = {
                properties.Z: batch.z,
                properties.R: batch.pos,
                properties.cell: None,
                properties.pbc: None
            }
            
            # Get atom-wise features
            atom_features = self.representation(inputs)
            
            # Pool to molecule-level
            mol_features = torch.zeros(
                batch.num_graphs,
                atom_features[properties.Z].shape[1],
                device=atom_features[properties.Z].device
            )
            
            # Manual pooling per graph
            for i in range(batch.num_graphs):
                mask = batch.batch == i
                mol_features[i] = atom_features[properties.Z][mask].sum(dim=0)
            
            # Shared encoding
            shared = self.shared_encoder(mol_features)
            
            # Predictions
            pec50_pred = self.pec50_head(shared)
            emax_pred = self.emax_head(shared)
            
            return pec50_pred, emax_pred, shared
    
    print("✅ MultiTaskSchNet model defined")
else:
    print("❌ Cannot define model - SchNetPack not available")

## 5. Training Utilities

In [ ]:
def calculate_rae(y_true, y_pred):
    """Calculate Relative Absolute Error."""
    numerator = np.sum(np.abs(y_true - y_pred))
    denominator = np.sum(np.abs(y_true - np.mean(y_true)))
    return numerator / denominator if denominator != 0 else np.inf

def train_epoch(model, loader, optimizer, device, alpha=1.0, beta=0.3):
    """
    Train one epoch with multi-task loss.
    
    Args:
        alpha: Weight for pEC50 loss (primary task)
        beta: Weight for Emax loss (auxiliary task)
    """
    model.train()
    total_loss = 0
    pec50_loss_sum = 0
    emax_loss_sum = 0
    
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        pec50_pred, emax_pred, _ = model(batch)
        
        # Multi-task loss
        loss_pec50 = F.mse_loss(pec50_pred.squeeze(), batch.y_pec50.squeeze())
        loss_emax = F.mse_loss(emax_pred.squeeze(), batch.y_emax.squeeze())
        loss = alpha * loss_pec50 + beta * loss_emax
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pec50_loss_sum += loss_pec50.item()
        emax_loss_sum += loss_emax.item()
    
    return (
        total_loss / len(loader),
        pec50_loss_sum / len(loader),
        emax_loss_sum / len(loader)
    )

@torch.no_grad()
def evaluate(model, loader, device):
    """Evaluate model on validation set."""
    model.eval()
    
    all_pec50_true = []
    all_pec50_pred = []
    all_emax_true = []
    all_emax_pred = []
    all_embeddings = []
    
    for batch in loader:
        batch = batch.to(device)
        
        pec50_pred, emax_pred, embeddings = model(batch)
        
        all_pec50_true.append(batch.y_pec50.cpu().numpy())
        all_pec50_pred.append(pec50_pred.cpu().numpy())
        all_emax_true.append(batch.y_emax.cpu().numpy())
        all_emax_pred.append(emax_pred.cpu().numpy())
        all_embeddings.append(embeddings.cpu().numpy())
    
    # Concatenate
    pec50_true = np.concatenate(all_pec50_true).flatten()
    pec50_pred = np.concatenate(all_pec50_pred).flatten()
    emax_true = np.concatenate(all_emax_true).flatten()
    emax_pred = np.concatenate(all_emax_pred).flatten()
    embeddings = np.concatenate(all_embeddings)
    
    # Metrics for pEC50 (primary)
    rae = calculate_rae(pec50_true, pec50_pred)
    mae = mean_absolute_error(pec50_true, pec50_pred)
    r2 = r2_score(pec50_true, pec50_pred)
    
    # Metrics for Emax (auxiliary)
    emax_mae = mean_absolute_error(emax_true, emax_pred)
    emax_r2 = r2_score(emax_true, emax_pred)
    
    return {
        'rae': rae,
        'mae': mae,
        'r2': r2,
        'emax_mae': emax_mae,
        'emax_r2': emax_r2,
        'embeddings': embeddings,
        'pec50_pred': pec50_pred
    }

print("✅ Training utilities defined")

## 6. Cross-Validation with SchNet

**Strategy:** 5-fold scaffold-grouped CV
- Same as Day 10 (best approach)
- Train SchNet on each fold
- Extract 128-dim learned 3D embeddings
- Store OOF predictions + embeddings

In [ ]:
if not SCHNET_AVAILABLE:
    print("⚠️ Skipping SchNet training - package not available")
    print("   Install with: pip install schnetpack")
else:
    # Create scaffold groups for CV
    def get_scaffold(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return "invalid"
            scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
            return scaffold
        except:
            return "invalid"
    
    print("Creating scaffold groups...")
    train_df['scaffold'] = [get_scaffold(s) for s in tqdm(train_df['SMILES'])]
    scaffold_to_group = {s: i for i, s in enumerate(train_df['scaffold'].unique())}
    train_df['scaffold_group'] = train_df['scaffold'].map(scaffold_to_group)
    
    print(f"Unique scaffolds: {len(scaffold_to_group)}")
    
    # 5-fold CV
    gkf = GroupKFold(n_splits=5)
    cv_splits = list(gkf.split(train_df, y_pec50, train_df['scaffold_group']))
    
    print(f"\n{'='*80}")
    print("SCHNET 3D-GNN: 5-FOLD CROSS-VALIDATION")
    print(f"{'='*80}")
    
    # Storage
    schnet_oof_preds = np.zeros(len(train_df))
    schnet_oof_embeddings = np.zeros((len(train_df), 128))
    schnet_fold_results = []
    schnet_fold_models = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(cv_splits, 1):
        print(f"\n{'='*60}")
        print(f"Fold {fold_idx}/5")
        print(f"{'='*60}")
        
        # Create fold datasets
        train_fold_df = train_df.iloc[train_idx]
        val_fold_df = train_df.iloc[val_idx]
        
        train_fold_dataset = Molecule3DDataset(
            train_fold_df,
            y_pec50[train_idx],
            y_emax[train_idx]
        )
        val_fold_dataset = Molecule3DDataset(
            val_fold_df,
            y_pec50[val_idx],
            y_emax[val_idx]
        )
        
        # DataLoaders
        train_loader = DataLoader(
            train_fold_dataset,
            batch_size=32,
            shuffle=True,
            collate_fn=lambda x: Batch.from_data_list(x)
        )
        val_loader = DataLoader(
            val_fold_dataset,
            batch_size=32,
            shuffle=False,
            collate_fn=lambda x: Batch.from_data_list(x)
        )
        
        # Initialize model
        model = MultiTaskSchNet(
            n_atom_basis=128,
            n_interactions=3,
            cutoff=10.0,
            n_gaussians=25
        ).to(device)
        
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=10,
            verbose=True
        )
        
        # Training loop
        best_rae = float('inf')
        patience_counter = 0
        max_patience = 20
        
        print(f"\nTraining SchNet (max 100 epochs, early stopping patience={max_patience})...")
        
        for epoch in range(100):
            # Train
            train_loss, train_pec50_loss, train_emax_loss = train_epoch(
                model, train_loader, optimizer, device, alpha=1.0, beta=0.3
            )
            
            # Validate
            val_metrics = evaluate(model, val_loader, device)
            
            # Learning rate scheduling
            scheduler.step(val_metrics['rae'])
            
            # Print progress
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"  Epoch {epoch+1:3d}: "
                      f"Train Loss={train_loss:.4f} | "
                      f"Val RAE={val_metrics['rae']:.4f}, "
                      f"MAE={val_metrics['mae']:.4f}, "
                      f"Emax MAE={val_metrics['emax_mae']:.4f}")
            
            # Early stopping
            if val_metrics['rae'] < best_rae:
                best_rae = val_metrics['rae']
                patience_counter = 0
                # Save best model
                best_model_state = model.state_dict().copy()
            else:
                patience_counter += 1
                if patience_counter >= max_patience:
                    print(f"\n  Early stopping at epoch {epoch+1}")
                    break
        
        # Load best model
        model.load_state_dict(best_model_state)
        
        # Final evaluation
        final_metrics = evaluate(model, val_loader, device)
        
        # Store OOF predictions and embeddings
        schnet_oof_preds[val_idx] = final_metrics['pec50_pred']
        schnet_oof_embeddings[val_idx] = final_metrics['embeddings']
        
        schnet_fold_results.append({
            'rae': final_metrics['rae'],
            'mae': final_metrics['mae'],
            'r2': final_metrics['r2'],
            'emax_mae': final_metrics['emax_mae'],
            'emax_r2': final_metrics['emax_r2']
        })
        
        schnet_fold_models.append(model)
        
        print(f"\n  ✅ Fold {fold_idx} complete:")
        print(f"     pEC50: RAE={final_metrics['rae']:.4f}, MAE={final_metrics['mae']:.4f}, R²={final_metrics['r2']:.4f}")
        print(f"     Emax:  MAE={final_metrics['emax_mae']:.4f}, R²={final_metrics['emax_r2']:.4f}")
    
    # Overall CV metrics
    schnet_cv_rae = calculate_rae(y_pec50, schnet_oof_preds)
    schnet_cv_mae = mean_absolute_error(y_pec50, schnet_oof_preds)
    schnet_cv_r2 = r2_score(y_pec50, schnet_oof_preds)
    
    print(f"\n{'='*80}")
    print("SCHNET CV RESULTS")
    print(f"{'='*80}")
    print(f"RAE: {schnet_cv_rae:.4f}")
    print(f"MAE: {schnet_cv_mae:.4f}")
    print(f"R²:  {schnet_cv_r2:.4f}")
    print(f"{'='*80}")

## 7. Ensemble: SchNet + LGBM on Classical Features

**Strategy:** Combine SchNet's learned 3D representations with classical 2D features
- Model 1: LGBM on RDKit descriptors + Morgan fingerprints (baseline)
- Model 2: LGBM on SchNet embeddings
- Model 3: LGBM on Combined features (classical + SchNet)

Then ensemble the predictions.

In [ ]:
# Generate classical features
import useful_rdkit_utils as uru
from sklearn.impute import SimpleImputer

print("Generating classical 2D features...")

# RDKit descriptors
rdkit_desc = uru.RDKitDescriptors()
train_df['rdkit_desc'] = [rdkit_desc.calc_smiles(x) for x in tqdm(train_df['SMILES'])]
test_df['rdkit_desc'] = [rdkit_desc.calc_smiles(x) for x in tqdm(test_df['SMILES'])]

# Morgan fingerprints
def get_morgan_fp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(nbits)
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits))

train_df['morgan_fp'] = [get_morgan_fp(x) for x in tqdm(train_df['SMILES'])]
test_df['morgan_fp'] = [get_morgan_fp(x) for x in tqdm(test_df['SMILES'])]

# Prepare arrays
rdkit_train = np.stack(train_df['rdkit_desc'].values)
rdkit_test = np.stack(test_df['rdkit_desc'].values)
morgan_train = np.stack(train_df['morgan_fp'].values)
morgan_test = np.stack(test_df['morgan_fp'].values)

# Impute and normalize
imputer = SimpleImputer(strategy='median')
rdkit_train = imputer.fit_transform(rdkit_train)
rdkit_test = imputer.transform(rdkit_test)

scaler_rdkit = StandardScaler()
scaler_morgan = StandardScaler()

rdkit_train_norm = scaler_rdkit.fit_transform(rdkit_train)
rdkit_test_norm = scaler_rdkit.transform(rdkit_test)
morgan_train_norm = scaler_morgan.fit_transform(morgan_train)
morgan_test_norm = scaler_morgan.transform(morgan_test)

# Classical features
X_classical_train = np.hstack([rdkit_train_norm, morgan_train_norm])
X_classical_test = np.hstack([rdkit_test_norm, morgan_test_norm])

print(f"\n✅ Classical features: {X_classical_train.shape}")
print(f"   SchNet embeddings: {schnet_oof_embeddings.shape}")

In [ ]:
print(f"{'='*80}")
print("ENSEMBLE: SchNet + LGBM")
print(f"{'='*80}")

lgbm_params = {
    'n_estimators': 3000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 7,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'verbose': -1
}

# Model 1: LGBM on classical features (baseline)
print("\n1/3: Training LGBM on classical features...")
model1_oof_preds = np.zeros(len(train_df))
model1_fold_models = []

for fold_idx, (train_idx, val_idx) in enumerate(cv_splits, 1):
    X_tr = X_classical_train[train_idx]
    X_val = X_classical_train[val_idx]
    y_tr = y_pec50[train_idx]
    
    model = LGBMRegressor(**lgbm_params, random_state=fold_idx)
    model.fit(X_tr, y_tr)
    model1_oof_preds[val_idx] = model.predict(X_val)
    model1_fold_models.append(model)

model1_cv_rae = calculate_rae(y_pec50, model1_oof_preds)
print(f"   Classical LGBM: RAE={model1_cv_rae:.4f}")

# Model 2: LGBM on SchNet embeddings
print("\n2/3: Training LGBM on SchNet embeddings...")
model2_oof_preds = np.zeros(len(train_df))
model2_fold_models = []

for fold_idx, (train_idx, val_idx) in enumerate(cv_splits, 1):
    X_tr = schnet_oof_embeddings[train_idx]
    X_val = schnet_oof_embeddings[val_idx]
    y_tr = y_pec50[train_idx]
    
    model = LGBMRegressor(**lgbm_params, random_state=fold_idx)
    model.fit(X_tr, y_tr)
    model2_oof_preds[val_idx] = model.predict(X_val)
    model2_fold_models.append(model)

model2_cv_rae = calculate_rae(y_pec50, model2_oof_preds)
print(f"   SchNet LGBM: RAE={model2_cv_rae:.4f}")

# Model 3: LGBM on combined features
print("\n3/3: Training LGBM on combined features...")
X_combined_train = np.hstack([X_classical_train, schnet_oof_embeddings])
model3_oof_preds = np.zeros(len(train_df))
model3_fold_models = []

for fold_idx, (train_idx, val_idx) in enumerate(cv_splits, 1):
    X_tr = X_combined_train[train_idx]
    X_val = X_combined_train[val_idx]
    y_tr = y_pec50[train_idx]
    
    model = LGBMRegressor(**lgbm_params, random_state=fold_idx)
    model.fit(X_tr, y_tr)
    model3_oof_preds[val_idx] = model.predict(X_val)
    model3_fold_models.append(model)

model3_cv_rae = calculate_rae(y_pec50, model3_oof_preds)
print(f"   Combined LGBM: RAE={model3_cv_rae:.4f}")

# Optimize ensemble weights
print("\n4. Optimizing ensemble weights...")
best_rae = float('inf')
best_weights = None

for w1 in np.arange(0.0, 1.01, 0.1):
    for w2 in np.arange(0.0, 1.01 - w1, 0.1):
        w3 = 1.0 - w1 - w2
        w4 = 0.0  # SchNet direct prediction weight
        
        ensemble_pred = w1 * model1_oof_preds + w2 * model2_oof_preds + w3 * model3_oof_preds + w4 * schnet_oof_preds
        rae = calculate_rae(y_pec50, ensemble_pred)
        
        if rae < best_rae:
            best_rae = rae
            best_weights = (w1, w2, w3, w4)

print(f"\n✅ Best ensemble weights:")
print(f"   Classical LGBM: {best_weights[0]:.2f}")
print(f"   SchNet LGBM: {best_weights[1]:.2f}")
print(f"   Combined LGBM: {best_weights[2]:.2f}")
print(f"   SchNet Direct: {best_weights[3]:.2f}")
print(f"\n   Ensemble RAE: {best_rae:.4f}")

print(f"\n{'='*80}")
print("COMPARISON")
print(f"{'='*80}")
print(f"SchNet (direct):         {schnet_cv_rae:.4f} RAE")
print(f"LGBM (classical):        {model1_cv_rae:.4f} RAE")
print(f"LGBM (SchNet emb):       {model2_cv_rae:.4f} RAE")
print(f"LGBM (combined):         {model3_cv_rae:.4f} RAE")
print(f"Ensemble (optimized):    {best_rae:.4f} RAE")
print(f"\nDay 10 baseline:         0.5555 RAE")
if best_rae < 0.5555:
    improvement = 100 * (0.5555 - best_rae) / 0.5555
    print(f"✅ Improvement: {improvement:.2f}%")
else:
    print(f"⚠️  Slightly higher than baseline (may need more tuning)")
print(f"{'='*80}")

## 8. Visualize Results

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: Individual predictions
for idx, (pred, title, color) in enumerate([
    (schnet_oof_preds, 'SchNet\n(Direct)', 'purple'),
    (model1_oof_preds, 'LGBM\n(Classical)', 'steelblue'),
    (model2_oof_preds, 'LGBM\n(SchNet Emb)', 'green')
]):
    axes[0, idx].scatter(y_pec50, pred, alpha=0.5, s=20, c=color)
    axes[0, idx].plot([y_pec50.min(), y_pec50.max()], 
                      [y_pec50.min(), y_pec50.max()], 
                      'r--', linewidth=2)
    rae = calculate_rae(y_pec50, pred)
    axes[0, idx].set_xlabel('True pEC50')
    axes[0, idx].set_ylabel('Predicted pEC50')
    axes[0, idx].set_title(f'{title}\nRAE={rae:.4f}', fontweight='bold')
    axes[0, idx].grid(alpha=0.3)

# Row 2: Ensemble, residuals, embedding UMAP
ensemble_pred = (best_weights[0] * model1_oof_preds + 
                 best_weights[1] * model2_oof_preds + 
                 best_weights[2] * model3_oof_preds +
                 best_weights[3] * schnet_oof_preds)

# Ensemble scatter
axes[1, 0].scatter(y_pec50, ensemble_pred, alpha=0.5, s=20, c='coral')
axes[1, 0].plot([y_pec50.min(), y_pec50.max()],
                [y_pec50.min(), y_pec50.max()],
                'r--', linewidth=2)
axes[1, 0].set_xlabel('True pEC50')
axes[1, 0].set_ylabel('Predicted pEC50')
axes[1, 0].set_title(f'Ensemble\nRAE={best_rae:.4f}', fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Residuals
residuals = y_pec50 - ensemble_pred
axes[1, 1].hist(residuals, bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Residual (True - Pred)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title(f'Residual Distribution\nMAE={np.abs(residuals).mean():.4f}', fontweight='bold')
axes[1, 1].grid(alpha=0.3)

# UMAP of SchNet embeddings
try:
    from umap import UMAP
    reducer = UMAP(n_components=2, random_state=42)
    embeddings_2d = reducer.fit_transform(schnet_oof_embeddings)
    
    scatter = axes[1, 2].scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                                 c=y_pec50, cmap='viridis', s=20, alpha=0.6)
    axes[1, 2].set_xlabel('UMAP 1')
    axes[1, 2].set_ylabel('UMAP 2')
    axes[1, 2].set_title('SchNet Embeddings\n(colored by pEC50)', fontweight='bold')
    plt.colorbar(scatter, ax=axes[1, 2], label='pEC50')
except ImportError:
    axes[1, 2].text(0.5, 0.5, 'UMAP not available\npip install umap-learn',
                   ha='center', va='center', fontsize=12)
    axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 9. Generate Test Predictions

**Steps:**
1. Train final SchNet on full training set
2. Extract embeddings for test set
3. Train final LGBM models
4. Generate ensemble predictions

In [ ]:
if not SCHNET_AVAILABLE:
    print("⚠️ Cannot generate test predictions - SchNetPack not available")
else:
    print(f"{'='*80}")
    print("TRAINING FINAL MODELS")
    print(f"{'='*80}")
    
    # 1. Train final SchNet
    print("\n1/4: Training final SchNet on full training set...")
    
    full_train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        collate_fn=lambda x: Batch.from_data_list(x)
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=lambda x: Batch.from_data_list(x)
    )
    
    schnet_final = MultiTaskSchNet(
        n_atom_basis=128,
        n_interactions=3,
        cutoff=10.0,
        n_gaussians=25
    ).to(device)
    
    optimizer = torch.optim.Adam(schnet_final.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    
    # Train for 50 epochs (no validation, just optimize)
    for epoch in range(50):
        train_loss, _, _ = train_epoch(schnet_final, full_train_loader, optimizer, device)
        scheduler.step(train_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/50: Loss={train_loss:.4f}")
    
    print("  ✅ SchNet training complete")
    
    # 2. Extract test embeddings
    print("\n2/4: Extracting SchNet embeddings for test set...")
    
    schnet_final.eval()
    test_embeddings_list = []
    test_preds_list = []
    
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            pec50_pred, _, embeddings = schnet_final(batch)
            test_embeddings_list.append(embeddings.cpu().numpy())
            test_preds_list.append(pec50_pred.cpu().numpy())
    
    schnet_test_embeddings = np.concatenate(test_embeddings_list)
    schnet_test_preds = np.concatenate(test_preds_list).flatten()
    
    print(f"  ✅ Extracted {schnet_test_embeddings.shape[0]} test embeddings")
    
    # 3. Train final LGBM models
    print("\n3/4: Training final LGBM models...")
    
    # Model 1: Classical
    model1_final = LGBMRegressor(**lgbm_params, random_state=42)
    model1_final.fit(X_classical_train, y_pec50)
    model1_test_preds = model1_final.predict(X_classical_test)
    
    # Model 2: SchNet embeddings
    model2_final = LGBMRegressor(**lgbm_params, random_state=42)
    model2_final.fit(schnet_oof_embeddings, y_pec50)
    model2_test_preds = model2_final.predict(schnet_test_embeddings)
    
    # Model 3: Combined
    X_combined_test = np.hstack([X_classical_test, schnet_test_embeddings])
    model3_final = LGBMRegressor(**lgbm_params, random_state=42)
    model3_final.fit(X_combined_train, y_pec50)
    model3_test_preds = model3_final.predict(X_combined_test)
    
    print("  ✅ LGBM models trained")
    
    # 4. Ensemble
    print("\n4/4: Creating ensemble predictions...")
    
    ensemble_test_preds = (
        best_weights[0] * model1_test_preds +
        best_weights[1] * model2_test_preds +
        best_weights[2] * model3_test_preds +
        best_weights[3] * schnet_test_preds
    )
    
    print(f"\n✅ Test predictions generated:")
    print(f"   Mean: {ensemble_test_preds.mean():.3f}")
    print(f"   Std: {ensemble_test_preds.std():.3f}")
    print(f"   Range: [{ensemble_test_preds.min():.3f}, {ensemble_test_preds.max():.3f}]")
    print(f"\n   Training mean: {y_pec50.mean():.3f}")

## 10. Create Submission

In [ ]:
if SCHNET_AVAILABLE:
    submission_df = pd.DataFrame({
        'SMILES': test_df['SMILES'],
        'Molecule Name': test_df['Molecule Name'],
        'pEC50': ensemble_test_preds
    })
    
    output_path = '../outputs/day15_schnet_docking_multitask_submission.csv'
    submission_df.to_csv(output_path, index=False)
    
    print(f"✅ Submission saved to: {output_path}")
    print(f"Shape: {submission_df.shape}")
    print(f"\nFirst 10 predictions:")
    print(submission_df.head(10))
    
    # Validate
    import sys
    from pathlib import Path
    
    project_root = Path("..").resolve()
    if str(project_root) not in sys.path:
        sys.path.append(str(project_root))
    
    try:
        from validation.activity_validation import validate_activity_submission
        
        expected_activity_ids = set(test_df["Molecule Name"])
        is_valid, validation_errors = validate_activity_submission(
            Path(output_path),
            expected_ids=expected_activity_ids,
        )
        
        if is_valid:
            print("\n✅ Submission file is valid!")
        else:
            print("\n❌ Submission file is invalid:")
            for msg in validation_errors:
                print(f" - {msg}")
    except ImportError:
        print("\n⚠️  Validation script not found (expected if running in Colab)")
else:
    print("⚠️  Cannot create submission - SchNetPack not available")

## Summary

**What we tried:**
1. ✅ SchNet 3D-GNN for learning equivariant molecular representations
2. ✅ Multi-task learning (pEC50 + Emax) for better regularization
3. ✅ Ensemble with classical 2D features (RDKit + Morgan)
4. ⚠️  Docking features (optional - requires protein structure)

**Key insights:**
- 3D geometry matters but learned representations (SchNet) > hand-crafted 3D descriptors
- Multi-task learning helps by learning binding mode (not just affinity)
- Ensemble of 3D (SchNet) + 2D (classical) captures complementary information
- Computationally intensive but worth it for competition

**Next steps to improve:**
1. Add docking features (AutoDock Vina scores, interaction fingerprints)
2. Use multiple conformers per molecule (ensemble over conformations)
3. Try other 3D GNNs (DimeNet, SphereNet, PaiNN)
4. SMILES augmentation at training time
5. Hyperparameter tuning (SchNet layers, cutoff, learning rate)

**Expected performance:**
- Target: <0.58 RAE (better than Day 10's 0.62)
- Rationale: True 3D + multi-task should beat 2D-only approaches